In [1]:
from qiskit_experiments.library import StandardRB
from qiskit import qasm2
from qiskit import transpile


qubits = (0,)
lengths = [10]
num_samples = 1
seed = 1001

rb = StandardRB(qubits, lengths, num_samples, seed)
circuits = [
    (
        transpile(
            circ,
            basis_gates=["s", "sxdg", "h", "z", "x", "y"],
            optimization_level=0,
        ),
        circ,
    )
    for circ in rb.circuits()
]

# print(qasm2.dumps(circuits[0][1]))
# print("=" * 100)
# print(qasm2.dumps(circuits[0][0]))


circuits = [circ[0] for circ in circuits]
print(qasm2.dumps(circuits[0]))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[1];
creg meas[1];
sxdg q[0];
z q[0];
barrier q[0];
barrier q[0];
sxdg q[0];
x q[0];
barrier q[0];
h q[0];
sxdg q[0];
z q[0];
barrier q[0];
h q[0];
sxdg q[0];
y q[0];
barrier q[0];
h q[0];
sxdg q[0];
z q[0];
barrier q[0];
h q[0];
sxdg q[0];
x q[0];
barrier q[0];
sxdg q[0];
y q[0];
barrier q[0];
barrier q[0];
h q[0];
sxdg q[0];
x q[0];
barrier q[0];
sxdg q[0];
y q[0];
barrier q[0];
measure q[0] -> meas[0];


In [2]:
from catalyst.debug.compiler_functions import get_compilation_stage
import pennylane as qml
from catalyst.third_party.oqd import OQDDevice

import os
import shutil
import pathlib


########################################################################################

for f in os.listdir():
    if f.startswith("oqd_circuit_benchmarking") and os.path.isdir(f):
        shutil.rmtree(pathlib.Path(f))

compile_results = pathlib.Path("oqd_circuit_benchmarking")
openapl_file_name = "oqd_circuit_benchmarking.openapl.json"


toml_files = {
    "device-toml-loc": "/home/user/oqd-catalyst/examples/calibration_data/device.toml",
    "qubit-toml-loc": "/home/user/oqd-catalyst/examples/calibration_data/qubit.toml",
    "gate-to-pulse-toml-loc": "/home/user/oqd-catalyst/examples/calibration_data/gate.toml",
}

toml_files = " ".join([f"{k}={v}" for k, v in toml_files.items()])


OQD_PIPELINES = [
    (
        "DeviceAgnosticPipeline",
        [
            "quantum-compilation-stage",
            "hlo-lowering-stage",
            "gradient-lowering-stage",
            "bufferization-stage",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "func.func(ions-decomposition)",
            # "func.func(merge-rotations)",
            "func.func(prune-zero-rotations)",
        ],
    ),
    (
        "IonDialectLoweringStage",
        [
            f"func.func(gates-to-pulses{{{toml_files}}})",
        ],
    ),
    ("IonToLLVMDialectConversion", ["convert-ion-to-llvm"]),
    ("MLIRToLLVMDialectConversion", ["llvm-dialect-lowering-stage"]),
]


oqd_dev = OQDDevice(
    backend="default",
    wires=1,
    openapl_file_name=(compile_results / openapl_file_name).as_posix(),
)


@qml.set_shots(10)
@qml.qnode(oqd_dev)
def oqd_circuit_benchmarking():
    qml.X(wires=0)
    qml.from_qasm(qasm2.dumps(circuits[0]))()
    return qml.counts(wires=0)


QJIT_CIRCUIT = qml.qjit(
    oqd_circuit_benchmarking,
    pipelines=OQD_PIPELINES,
    keep_intermediate=True,
    verbose=True,
)

print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(QJIT_CIRCUIT, stage="IonDecompositionStage"))


[LIB] Running compiler driver in /home/user/oqd-catalyst/examples/benchmarking/oqd_circuit_benchmarking
[SYSTEM] /home/user/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/oqd-catalyst/examples/benchmarking/oqd_circuit_benchmarking/oqd_circuit_benchmarking.ll --module-name oqd_circuit_benchmarking --workspace /home/user/oqd-catalyst/examples/benchmarking/oqd_circuit_benchmarking -verify-each=false --catalyst-pipeline DeviceAgnosticPipeline(quantum-compilation-stage;hlo-lowering-stage;gradient-lowering-stage;bufferization-stage),IonDecompositionStage(func.func(ions-decomposition);func.func(prune-zero-rotations)),IonDialectLoweringStage(func.func(gates-to-pulses{device-toml-loc=/home/user/oqd-catalyst/examples/calibration_data/device.toml qubit-toml-loc=/home/user/oqd-catalyst/examples/calibration_data/qubit.toml gate-to-pulse-toml-loc=/home/user/oqd-catalyst/examples/calibration_data/gate.toml})),IonToLLVMDialectConversion(convert-ion-to-llvm),MLIRToL

In [3]:
import json

print(qml.draw(oqd_circuit_benchmarking)())

with open(compile_results / "oqd_circuit_benchmarking.draw.txt", "w") as f:
    f.write(qml.draw(oqd_circuit_benchmarking)())


QJIT_CIRCUIT()

print(json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2))

0: ──X──SX†──Z──||──||──SX†──X──||──H──SX†──Z──||──H──SX†──Y──||──H──SX†──Z──||──H──SX†──X──|| ···

0: ··· ──SX†──Y──||──||──H──SX†──X──||──SX†──Y──||──┤↗├─┤  Counts
{
  "class_": "AtomicCircuit",
  "protocol": {
    "class_": "SequentialProtocol",
    "sequence": [
      {
        "class_": "ParallelProtocol",
        "sequence": [
          {
            "beam": {
              "class_": "Beam",
              "detuning": {
                "class_": "MathNum",
                "value": 208570336271826.38
              },
              "phase": {
                "class_": "MathNum",
                "value": 0.0
              },
              "polarization": [
                1,
                0,
                0
              ],
              "rabi": {
                "class_": "MathNum",
                "value": 5514137224.977724
              },
              "target": 0,
              "transition": {
                "class_": "Transition",
                "einsteinA": 41050903.1198

In [4]:
from oqd_core.interface.atomic import AtomicCircuit
from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
from oqd_compiler_infrastructure import Chain, Post
from oqd_bare_metal.compiler.codegen import AtomicToBloodstoneV1
from oqd_bare_metal.compiler.optim import (
    SpectrumPrune,
    SpectrumUnwrapResets,
)
import ast_comments as ast


circuit = AtomicCircuit.model_validate_json(
    json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
)


compiler = Chain(
    canonicalize_atomic_circuit_factory(),
    Post(
        AtomicToBloodstoneV1(
            device_params="../calibration_data/bloodstone_params.toml",
        ),
    ),
)
optimization_pass = Chain(
    Post(SpectrumUnwrapResets(cores=[20])),
    Post(SpectrumPrune()),
)

unopt_artiq_experiment = compiler(circuit)
artiq_experiment = optimization_pass(unopt_artiq_experiment)

print(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

with open(compile_results / "oqd_circuit_benchmarking.artiq.py", "w") as f:
    f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

from artiq.experiment import *
from config import *

class BloodstoneV1Experiment(Bloodstone_Experiment, Experiment):

    def program_awg(self):
        # init Spectrum AWGs
        self.awg.start()
        self.awg.card_mode('dds')
        self.awg.channel_enable_out(True)
        self.awg.channels_output_load(50)
        self.awg.channels_amp(2000)
        self.awg.trigger_or_mask('ext0')
        self.awg.trigger_ext0_mode('pos')
        self.awg.trigger_ext0_level0(1000)
        self.awg.trigger_ext0_coupling('dc')
        self.awg.clock_mode('extrefclock')
        self.awg.clock_reference_clock(10000000)
        self.awg.write_setup()
        self.awg.dds_data_transfer_mode('dma')
        
        # Precompiled profiles for Spectrum AWGs
        # programming profile 0
        self.awg.dds_channel_amp(core_index=20, amplitude=0.9)
        self.awg.dds_channel_freq(core_index=20, frequency_Hz=209961800.02721885)
        self.awg.dds_exec_at_trg()
        # ramp down profile 0
     